# Build AI Agents with Persistent Memory Using Amazon Bedrock AgentCore

In this notebook, you build an IT help desk agent that remembers employee device information, past issues, and preferences across multiple sessions using Amazon Bedrock AgentCore Memory.

# Task 1: Create a baseline agent

In this task, you build a basic IT help desk agent using the Strands Agents framework. This establishes a baseline that has **no memory**: a simple Strands agent with three local tools and no memory.

### Task 1.1: Setup and import libraries

Import the libraries you need for the agent, AgentCore Memory, and the Strands Agents framework.

In [ ]:
import boto3
import time
from datetime import datetime
from boto3.session import Session

# AgentCore imports
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

# Strands imports
from strands import Agent
from strands.models import BedrockModel

# Local tools for the IT help desk agent
from lab_helpers.helpdesk_agent import (
    check_system_status,
    lookup_knowledge_base,
    create_ticket,
    SYSTEM_PROMPT,
    MODEL_ID,
)
from lab_helpers.utils import get_ssm_parameter, put_ssm_parameter

# Setup
boto_session = Session()
REGION = boto_session.region_name
EMPLOYEE_ID = "employee_001"  # The actor_id for the employee using the help desk

print("✅ Libraries imported successfully")
print(f"📍 Region: {REGION}")
print(f"👤 Employee ID (actor_id): {EMPLOYEE_ID}")

### Task 1.2: Examine the IT help desk tools

Before building the agent, open `lab_helpers/helpdesk_agent.py` and review the three tools:

- **`check_system_status(service_name)`** — Returns the operational status of internal services (VPN, email, SSO, etc.)
- **`lookup_knowledge_base(issue_category, keywords)`** — Returns remediation steps from the internal KB
- **`create_ticket(priority, category, description)`** — Creates a support ticket and returns a ticket ID

These tools return mock data so you can focus on learning AgentCore Memory without needing live infrastructure. Each tool simulates a realistic response — `check_system_status` returns randomized service health, `lookup_knowledge_base` returns pre-written remediation steps, and `create_ticket` generates a mock ticket ID. In a real deployment, these would be backed by your actual monitoring dashboards, knowledge base APIs, and ticketing systems.

Also notice the `SYSTEM_PROMPT` defined in `helpdesk_agent.py`. The system prompt is what gives the agent its persona and operational boundaries — it tells the model that it is an IT help desk assistant, describes the tools it has access to, and sets behavioral guidelines (like always checking system status before suggesting fixes).

### Task 1.3: Build the baseline agent

Create a Strands agent with the Bedrock foundation model and the three IT help desk tools. This agent has no memory — each conversation starts from scratch.

In [ ]:
# Initialize the Bedrock foundation model
model = BedrockModel(
    model_id=MODEL_ID,
    temperature=0.3,
    region_name=REGION,
)

# Create a baseline agent with local tools only (no memory)
baseline_agent = Agent(
    model=model,
    tools=[check_system_status, lookup_knowledge_base, create_ticket],
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Baseline IT help desk agent ready")
print("📋 Tools: check_system_status, lookup_knowledge_base, create_ticket")
print("⚠️  Note: This agent has no memory of prior conversations")

### Task 1.4: Test the baseline agent

Test the agent with a sample employee query. Notice that the agent has no knowledge of the employee — no device info, no history, no preferences.

In [ ]:
print("💬 Testing the baseline agent (no memory)...\n")
response = baseline_agent(
    "My VPN keeps disconnecting. What should I try?"
)
print("\n" + "=" * 60 + "\n")

**Task complete.** You have a baseline agent with no memory. The agent has no knowledge of you — no device info, no history, no preferences — so any context it needs has to come from the prompt itself. 

In the next task, you add AgentCore Memory so the agent can remember context across sessions.

# Task 2: Create AgentCore Memory

AgentCore Memory is a managed service that gives your agent persistent memory across sessions. It provides built-in long-term memory strategies that automatically extract different types of information from conversations. This lab focuses on three of them:

| Strategy | API key | What it captures | Example |
|---|---|---|---|
| **Semantic** | `semanticMemoryStrategy` | Factual information and contextual knowledge | "Employee uses a MacBook Pro M2 running macOS Sonoma" |
| **User Preference** | `userPreferenceMemoryStrategy` | Preferences, choices, and behavioral patterns | "Prefers email follow-ups over chat" |
| **Summary** | `summaryMemoryStrategy` | Condensed session summaries (key topics, tasks, decisions) | "Employee reported VPN disconnections; given KB steps; issue resolved" |

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Note:** AgentCore Memory provides additional built-in strategies beyond these three, along with custom and self-managed options. You can find the full list in the [AgentCore Memory strategies documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory-strategies.html).

### Task 2.1: Create the AgentCore Memory resource

Create a memory resource configured with the three built-in strategies. Creation takes 2-3 minutes.

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Note:** If the memory resource already exists, the code retrieves the existing ID instead of creating a new one.

In [ ]:
# Initialize the memory client
memory_client = MemoryClient(region_name=REGION)
MEMORY_NAME = "HelpDeskMemory"
MEMORY_ID_SSM_PARAM = "/app/helpdesk/agentcore/memory_id"


def create_or_get_memory_resource():
    """Create a memory resource or return the existing one if it already exists."""
    try:
        # Check if a memory resource already exists from a prior run
        memory_id = get_ssm_parameter(MEMORY_ID_SSM_PARAM)
        memory_client.gmcp_client.get_memory(memoryId=memory_id)
        print(f"✅ Using existing memory resource: {memory_id}")
        return memory_id
    except Exception:
        # Create a new memory resource with the three built-in strategies
        strategies = [
            {
                StrategyType.SEMANTIC.value: {
                    "name": "HelpDeskFacts",
                    "description": "Extracts factual information about employees and their devices",
                    "namespaces": ["helpdesk/employee/{actorId}/facts"],
                }
            },
            {
                StrategyType.USER_PREFERENCE.value: {
                    "name": "HelpDeskPreferences",
                    "description": "Captures employee preferences for support interactions",
                    "namespaces": ["helpdesk/employee/{actorId}/preferences"],
                }
            },
            {
                StrategyType.SUMMARY.value: {
                    "name": "HelpDeskSummaries",
                    "description": "Creates condensed summaries of each support session",
                    "namespaces": ["helpdesk/summaries/{actorId}/{sessionId}"],
                }
            },
        ]
        print("⏳ Creating AgentCore Memory resource (2-3 minutes)...")
        response = memory_client.create_memory_and_wait(
            name=MEMORY_NAME,
            description="Memory for the IT help desk agent",
            strategies=strategies,
            event_expiry_days=90,  # Short-term events expire after 90 days (long-term records do not expire based on this)
        )
        memory_id = response["id"]
        put_ssm_parameter(MEMORY_ID_SSM_PARAM, memory_id)
        print(f"✅ Memory resource created: {memory_id}")
        return memory_id


memory_id = create_or_get_memory_resource()

**Note:** As you can see from the code, AgentCore Memory uses **namespaces** to organize memories. You can scope namespaces by `{actorId}` (the employee) and `{sessionId}` (a specific conversation). Semantic and User Preference records, accumulate stable facts and preferences about an actor across sessions, so their namespaces are typically scoped by `{actorId}` only. Summary uses `{sessionId}` because summary records are generated and maintained at the session level — they condense the topics of a single conversation.

### Task 2.2: Inspect the memory strategies

Next, you view the three strategies configured on your memory resource. Each strategy has a type and a namespace template that determines where memories are stored.

In [ ]:
# List the strategies configured on the memory resource
strategies = memory_client.get_memory_strategies(memory_id)

print("📋 Configured memory strategies:\n")
for strategy in strategies:
    print(f"  Type: {strategy['type']}")
    print(f"  Name: {strategy['name']}")
    print(f"  Namespace: {strategy['namespaces'][0]}")
    print()

**Task complete.** You created an AgentCore Memory resource with the three built-in strategies used in this lab (Semantic, User Preference, and Summary). Memory is now ready — but your agent is not yet using it. In Task 3, you wire the memory into the agent.

# Task 3: Integrate memory with the agent

In this task, you connect your memory resource to the Strands agent using `AgentCoreMemorySessionManager`. The session manager handles:

- **Retrieval** — Before the agent responds, the session manager retrieves recent conversation events (short-term memory) and relevant long-term records from the configured strategy namespaces, then adds both to the agent's context
- **Storage** — After the agent responds, the conversation turn is saved as an event in short-term memory. Long-term strategies then asynchronously extract facts, preferences, and summaries from those events

### Task 3.1: Configure the memory session manager

The `AgentCoreMemorySessionManager` ties together:
- The memory resource (via `memory_id`)
- The employee (via `actor_id`) — this is how memory persists across sessions for the same person
- The current session (via `session_id`) — a unique ID for this conversation

The combination of `actor_id` + `session_id` is how AgentCore Memory knows which memories belong to whom, and which conversation is happening now.

In [ ]:
from bedrock_agentcore.memory.integrations.strands.config import (
    AgentCoreMemoryConfig,
    RetrievalConfig,
)
from bedrock_agentcore.memory.integrations.strands.session_manager import (
    AgentCoreMemorySessionManager,
)

# Create a unique session ID for this conversation
SESSION_ID_1 = f"session-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

# Configure memory to connect to your resource
memory_config = AgentCoreMemoryConfig(
    memory_id=memory_id,
    actor_id=EMPLOYEE_ID,
    session_id=SESSION_ID_1,
)

# Create the session manager - this handles automatic retrieval and storage
session_manager_1 = AgentCoreMemorySessionManager(
    agentcore_memory_config=memory_config,
    region_name=REGION,
)

print(f"✅ Memory session manager configured")
print(f"   Memory ID: {memory_id}")
print(f"   Actor ID (employee): {EMPLOYEE_ID}")
print(f"   Session ID: {SESSION_ID_1}")
print(f"\n💡 actor_id identifies WHO is talking; session_id identifies WHICH conversation.")
print(f"   Together they determine where events are stored and what gets retrieved.")

### Task 3.2: Create the memory-enabled agent

Rebuild the agent with the session manager attached. This is the same agent as before, but now with persistent memory.

In [ ]:
# Create the agent with memory integration
memory_agent = Agent(
    model=model,
    tools=[check_system_status, lookup_knowledge_base, create_ticket],
    system_prompt=SYSTEM_PROMPT,
    session_manager=session_manager_1,  # Wires memory in — handles storing events and retrieving context
)

print("✅ Memory-enabled IT help desk agent created")
print("🧠 Agent will automatically retrieve context and save interactions")

**Task complete.** Your agent now has persistent memory wired in. In Task 4 you use it across multiple sessions to demonstrate cross-session recall.

But first — let's prove that short-term memory is already working.

### Task 3.3: Prove short-term memory works

AgentCore Memory has two tiers. The first is **short-term memory**: every conversation turn — each user message, agent response, and tool invocation — is stored as an immutable, timestamped **event** in AgentCore's managed store, organized by `actor_id` (who is talking) and `session_id` (which conversation). The `AgentCoreMemorySessionManager` handles this automatically — every time the agent processes a message, it calls `CreateEvent` behind the scenes.

The conversation history is preserved even if the agent process restarts — as long as the `session_id` stays the same.

To prove this, you will:
1. Tell the agent something specific
2. Create a **completely new** agent instance
3. Ask the new agent to recall what you said

If short-term memory is working, the new agent will know — because the session manager retrieves stored events from AgentCore.

In [ ]:
# Step 1: Tell the agent something specific
print("💬 Sharing information with the agent...\n")
print("-" * 60)
response = memory_agent(
    "Just so you know, my desk is in Building 4, floor 3, "
    "and my employee badge number is 7742."
)
print("\n" + "=" * 60)

In [ ]:
# Step 2: Create a BRAND NEW agent instance — same session_id, same actor_id.
# We also build a fresh session manager so the new agent does not
# share the same in-memory session_manager object as `memory_agent` — that way
# the only link between the two agents is the AgentCore-managed session state.
fresh_session_manager = AgentCoreMemorySessionManager(
    agentcore_memory_config=AgentCoreMemoryConfig(
        memory_id=memory_id,
        actor_id=EMPLOYEE_ID,
        session_id=SESSION_ID_1,  # Same session_id — that's the key
    ),
    region_name=REGION,
)

fresh_agent = Agent(
    model=model,
    tools=[check_system_status, lookup_knowledge_base, create_ticket],
    system_prompt=SYSTEM_PROMPT,
    session_manager=fresh_session_manager,
)

# Step 3: Ask the new agent to recall what you said
print("💬 Asking a FRESH agent instance (no Python history) to recall...\n")
print("-" * 60)
response = fresh_agent(
    "What building and floor is my desk on, and what's my badge number?"
)
print("\n" + "=" * 60)

**What just happened?**

The `fresh_agent` is a completely new Python object — it has no `messages` list, no conversation history from the previous cell. Anything it can say about your desk location or badge number had to come from outside the process.

That's short-term memory at work: the `session_manager` stored your previous turn as an **event** in AgentCore's managed store, and when `fresh_agent` initialized with the same `session_id`, it retrieved those events automatically.

**Key distinction to keep in mind:**
- **Short-term memory** is persistent but scoped to a single `session_id`. A new session won't see these events.
- **Long-term memory** (which you'll see in Task 4) persists across sessions — it follows the `actor_id` regardless of which session is active.

# Task 4: Demonstrate cross-session memory

In this task, you see what happens when you start a completely new session:

1. **Session 1** — Have a conversation where the employee shares device info and preferences. Memory is saved automatically.
2. **Session 2** — Start a completely new session (new `session_id`, same `actor_id`). Watch the agent recall context from Session 1 without being told.
3. **Try it yourself** — Write your own queries and see personalized responses.

This is the persistence that transforms a stateless agent into a context-aware assistant.

### Task 4.1: Session 1 — Establish context

In this session, the employee reports a VPN issue and shares details about their device and communication preferences. The agent resolves the issue, and behind the scenes, AgentCore Memory extracts facts and preferences into long-term memory.

In [ ]:
print("💬 SESSION 1 — TURN 1\n")
print("-" * 60)
response = memory_agent(
    "Hi, my VPN keeps disconnecting when I switch between the office and home. "
    "I'm on a MacBook Pro M2 running macOS Sonoma, using the GlobalProtect client. "
    "Also, I prefer email follow-ups over chat or phone calls."
)
print("\n" + "=" * 60 + "\n")

In [ ]:
print("💬 SESSION 1 — TURN 2\n")
print("-" * 60)
response = memory_agent(
    "Can you check the status of the VPN service?"
)
print("\n" + "=" * 60 + "\n")

In [ ]:
print("💬 SESSION 1 — TURN 3\n")
print("-" * 60)
response = memory_agent(
    "Thanks. Can you look up the KB article for this networking issue?"
)
print("\n" + "=" * 60 + "\n")

Short-term memory kept the conversation coherent across Turns 1-3. But it's scoped to a single session — start a new session, and the agent has no idea who you are.

In the next task, you see how **Long-term memory** solves this.

### Task 4.2: Wait for and verify memory extraction

**Long-term memory** solves the single-session limitation. The strategies you configured asynchronously extract structured records (facts, preferences, summaries) from short-term events. These records persist by `actor_id`, so when the same employee returns in a new session, the agent can retrieve them — no repetition needed.

That extraction is happening now in the background. Next, you'll wait for it to complete and then test cross-session recall. The cell below waits 90 seconds for the initial extraction, then polls the Semantic and User Preference namespaces to confirm that records exist before you proceed to Session 2.

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Note:** If the cell finishes with a warning that records are not yet available, wait 30 seconds and re-run it.

In [ ]:
print("⏳ Waiting 90 seconds for initial memory extraction...")
time.sleep(90)

# Verify that long-term memory extraction has actually completed
print("\n🔍 Verifying memory extraction completed...\n")

max_wait = 120  # additional seconds to wait if needed
poll_interval = 10
elapsed = 0
facts_ready = False

while elapsed < max_wait:
    response = memory_client.list_memory_records(
        memory_id=memory_id,
        namespace=f"helpdesk/employee/{EMPLOYEE_ID}/facts",
    )
    records = response.get("memoryRecordSummaries", [])
    if records:
        print(f"✅ Semantic memory extraction confirmed — {len(records)} record(s) found")
        facts_ready = True
        break
    print(f"   Still waiting... ({elapsed}s elapsed)")
    time.sleep(poll_interval)
    elapsed += poll_interval

if not facts_ready:
    print("⚠️  Extraction may still be in progress. Wait 30 seconds and re-run this cell.")

# Also check preferences
response = memory_client.list_memory_records(
    memory_id=memory_id,
    namespace=f"helpdesk/employee/{EMPLOYEE_ID}/preferences",
)
pref_records = response.get("memoryRecordSummaries", [])
if pref_records:
    print(f"✅ User Preference extraction confirmed — {len(pref_records)} record(s) found")
else:
    print("⚠️  User Preference records not yet available — they may take a bit longer")

print("\n📋 Ready for Session 2")

### Task 4.3: Session 2 — Demonstrate cross-session recall

Now start a completely new session with a new `session_id` but the same `actor_id` (employee). The new agent instance has **no conversation history from Session 1** in short-term memory — but with a `retrieval_config` pointed at the long-term memory namespaces, the session manager automatically pulls relevant facts and preferences into the agent's context before each turn.

To make the recall observable, the employee explicitly asks the agent to recall their device, OS, and preferred follow-up method — information that was only shared in Session 1. With long-term memory wired up, the agent can answer based on retrieved records without the employee repeating that information.

In [ ]:
# New session ID - this is a fresh conversation
SESSION_ID_2 = f"session-{datetime.now().strftime('%Y%m%d-%H%M%S')}-new"

# New memory config with new session_id (same actor_id).
# This time we pass retrieval_config — it tells the session manager which long-term
# memory namespaces to search before each user turn.
memory_config_2 = AgentCoreMemoryConfig(
    memory_id=memory_id,
    actor_id=EMPLOYEE_ID,  # Same employee
    session_id=SESSION_ID_2,  # New conversation
    retrieval_config={
        f"helpdesk/employee/{EMPLOYEE_ID}/facts": RetrievalConfig(
            top_k=5, relevance_score=0.2
        ),
        f"helpdesk/employee/{EMPLOYEE_ID}/preferences": RetrievalConfig(
            top_k=5, relevance_score=0.2
        ),
    },
)

session_manager_2 = AgentCoreMemorySessionManager(
    agentcore_memory_config=memory_config_2,
    region_name=REGION,
)

# New agent instance - no conversation history from Session 1 in short-term memory
memory_agent_session2 = Agent(
    model=model,
    tools=[check_system_status, lookup_knowledge_base, create_ticket],
    system_prompt=SYSTEM_PROMPT,
    session_manager=session_manager_2,
)

print(f"✅ Session 2 agent ready")
print(f"   Actor ID: {EMPLOYEE_ID} (same employee)")
print(f"   Session ID: {SESSION_ID_2} (new conversation)")

In [ ]:
print("💬 SESSION 2 — Testing cross-session recall\n")
print("-" * 60)
print("Employee asks an ambiguous follow-up without repeating device info:")
print()
response = memory_agent_session2(
    "The VPN issue is back. Can you remind me what device and OS I'm on, "
    "and what's the best way to follow up with me?"
)
print("\n" + "=" * 60 + "\n")

**What just happened?**

Before generating its response, the session manager retrieved long-term memory records from the namespaces in `retrieval_config`:

- `helpdesk/employee/employee_001/facts` (Semantic) — facts extracted from Session 1
- `helpdesk/employee/employee_001/preferences` (User Preference) — preferences extracted from Session 1

Whatever records were extracted are injected into the agent's context, so the agent can draw on them without the employee repeating anything.

### Task 4.4: Try it yourself

Continue the conversation in Session 2. Try queries that test different aspects of memory:

- Ask about a different system (e.g., "Can you check the status of the collaboration tools?")
- Ask a follow-up that requires device context (e.g., "What hardware diagnostics can I run?")
- Ask for a summary of what the agent knows about you

Write your own query in the cell below and observe how memory shapes the response.

In [ ]:
# Write your own query to test memory recall
# Example ideas:
#   - "Can you check the status of SSO?"
#   - "What do you know about my setup?"
#   - "Create a ticket for me about the recurring VPN issue"

your_query = "Can you create a support ticket for my VPN issue? Follow up with me the way I prefer."

print("💬 YOUR QUERY\n")
print("-" * 60)
response = memory_agent_session2(your_query)
print("\n" + "=" * 60 + "\n")

**Task complete.** You exercised cross-session memory — the same employee was used across two distinct sessions, with retrieval pulling extracted long-term records into the new session's context.

### Task 4.5: Inspect what each strategy extracted

Each strategy processes the same conversation differently. The cell below reads directly from each strategy's namespace using the AgentCore Memory SDK and prints the raw records.

- **Semantic** records are standalone facts about the actor
- **User Preference** records are behavioral patterns and stated preferences
- **Summary** records are session-level overviews — their value grows with longer conversations where a condensed digest saves the agent from replaying the full event history

In [ ]:
# Retrieve and display memory records from each strategy namespace
namespaces_to_check = [
    (f"helpdesk/employee/{EMPLOYEE_ID}/facts", "SEMANTIC"),
    (f"helpdesk/employee/{EMPLOYEE_ID}/preferences", "USER PREFERENCE"),
    (f"helpdesk/summaries/{EMPLOYEE_ID}/{SESSION_ID_1}", "SUMMARY"),
]


def extract_record_text(record):
    """Pull human-readable text out of a memory record summary."""
    content = record.get("content", {})
    if isinstance(content, dict):
        return content.get("text") or str(content)
    return str(content)


print("🧠 Long-term memory records by strategy:\n")
print("=" * 60)

for namespace, strategy_name in namespaces_to_check:
    print(f"\n📂 {strategy_name} — namespace: {namespace}\n")
    try:
        # list_memory_records returns the raw boto3 response; the records live
        # inside the memoryRecordSummaries key.
        response = memory_client.list_memory_records(
            memory_id=memory_id,
            namespace=namespace,
        )
        records = response.get("memoryRecordSummaries", [])
        if records:
            for i, record in enumerate(records, 1):
                text = extract_record_text(record)
                print(f"  [{i}] {text}")
        else:
            print("  (no records yet — extraction may still be in progress)")
    except Exception as e:
        print(f"  ⚠️ Could not retrieve: {e}")
    print("-" * 60)

print("\n💡 Notice how the same conversation produced different types of memories")
print("   depending on the strategy:")
print("   • Semantic = what the agent knows (facts)")
print("   • User Preference = how the user likes things done (personalization)")
print("   • Summary = what happened in the session (context for follow-ups)")

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Note:** If any namespace shows no records, wait 30 seconds and re-run the cell. Long-term memory extraction is asynchronous and may take additional time.

## Summary

You built an IT help desk agent with persistent memory using Amazon Bedrock AgentCore Memory. Specifically, you:

- ✅ Created a Strands agent with three IT help desk tools (baseline without memory)
- ✅ Created an AgentCore Memory resource with three built-in strategies — Semantic, User Preference, and Summary.
- ✅ Integrated memory into the agent using `AgentCoreMemorySessionManager`
- ✅ Proved that short-term memory persists beyond any single agent instance by recalling context through a fresh agent object
- ✅ Demonstrated cross-session retrieval: Semantic and User Preference records from Session 1 were retrieved into Session 2, without the employee repeating their context
- ✅ Explored what each strategy extracts from the same conversation

Return to the main lab instructions to explore the AgentCore Memory dashboard and finish the lab.